<!-- field-paper-role-banner -->
> 🟦 **실습본** — TODO를 직접 구현하고 assertion을 통과시키세요. 정답본은 같은 위치의 필요한 부분만 확인합니다.

# 자연어 처리와 LLM · 09. RAG

## Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks

- **원 논문:** [Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks](https://arxiv.org/abs/2005.11401)
- **저자 / 발표:** Patrick Lewis et al. · NeurIPS 2020 (2020)
- **난이도 / 예상 시간:** 중급 · 약 120분
- **선수 지식:** Python, PyTorch, 확률·선형대수·sequence model의 기초
- **로컬 학습 데이터:** `data/field_curriculum/nlp_corpus.json`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** retriever distribution과 RAG-Sequence marginalization을 재현한다.

**축소하거나 생략한 부분:** DPR+BART+Wikipedia 21M passages 대신 TF-IDF와 로컬 5문서를 사용한다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §2.1 Eq. (1): pη(z|x) | Task 1 softmax similarity | probability sum |
| §2.2 Eq. (2): RAG-Sequence | Task 2 marginalization | finite NLL |
| §2.2 Eq. (3): RAG-Token | Task 2 설명/비교 | per-token distinction |
| Fig. 1: retriever+generator | Task 3 | recall@1/cited answer |

## 핵심 아이디어와 수식

$$p_{RAG\text{-}Seq}(y\mid x)=\sum_{z\in top\text{-}k}p_{\eta}(z\mid x)p_{\theta}(y\mid x,z)$$

**기호 해설:** z는 문서, p_eta는 retriever, p_theta는 document-conditioned generator likelihood다.

**코드 대응:** Task 1이 TF-IDF retriever, Task 2가 RAGSequence class, Task 3이 NLL/recall을 구현한다.

**입력과 출력 shape:** query vectors [3,V], document vectors [5,V] → p(z|x) [3,5] → answer prob [3].

**포트폴리오 구현 범위:** neural DPR/BART 대신 결정적 surrogate를 쓰지만 latent-document marginalization은 정확하다.

각 단계는 데이터 전처리와 shape 확인부터 논문 고유 class, `forward`, 목적함수,
`train_step`, 평가까지 이어진다. 핵심 수식을 한 번의 고수준 호출로 감추지 않고
중간 tensor를 확인할 수 있게 분리한다.

In [ ]:
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
import json
import math
import re
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(20260814)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device


def locate(relative_path):
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(relative_path)


dataset_path = locate("data/field_curriculum/nlp_corpus.json")
corpus = json.loads(dataset_path.read_text(encoding="utf-8"))
sentences = corpus["sentences"]
translation_pairs = corpus["translation_pairs"]
documents = corpus["documents"]
queries = corpus["queries"]
SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>", "<mask>", "<sentinel>"]
sentence_words = {word.lower() for sentence in sentences for word in sentence.split()}
translation_words = {
    word.lower()
    for pair in translation_pairs
    for side in ("source", "target")
    for word in pair[side].split()
}
words = sorted(sentence_words | translation_words)
itos = SPECIAL_TOKENS + words
stoi = {word: index for index, word in enumerate(itos)}
PAD, BOS, EOS, MASK, SENTINEL = range(5)
VOCAB_SIZE = len(itos)


def encode(text, length=4):
    token_ids = [stoi[word.lower()] for word in text.split()]
    token_ids = token_ids[:length]
    return token_ids + [PAD] * (length - len(token_ids))


sentence_ids = torch.tensor([encode(text) for text in sentences])
sentiment_labels = torch.tensor(corpus["sentiment_labels"], dtype=torch.long)
source_ids = torch.tensor([encode(pair["source"], 3) for pair in translation_pairs])
target_ids = torch.tensor([encode(pair["target"], 3) for pair in translation_pairs])
sentence_ids, sentiment_labels = ACCELERATOR.move(
    sentence_ids,
    sentiment_labels,
)
source_ids, target_ids = ACCELERATOR.move(source_ids, target_ids)
assert sentence_ids.shape == (360, 4)
assert source_ids.shape == target_ids.shape == (60, 3)
print(ACCELERATOR.summary())
print(dataset_path.name, f"sentences={len(sentences)}, vocab={VOCAB_SIZE}")

### 구현 단계 1 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §2.1 Eq. (1): pη(z|x)
- **노트북에서 구현할 부분:** Task 1 softmax similarity
- **구현 이유:** 원문의 확률 분해나 tensor 변환을 작은 입력에서 직접
  관찰하고, 고수준 API 뒤에 핵심 알고리즘을 숨기지 않기 위해서다.
- **완료 증거:** probability sum

아래 TODO 전에 입력 token/feature shape와 수식의 축을 먼저 확인한다.

In [ ]:
# TODO 1: 설정, tokenizer/count vector, TF-IDF retriever 분포를 구현하세요.
@dataclass
class RAGConfig:
    pass


def terms(text):
    raise NotImplementedError


def count_vector(text, vocabulary, term_to_id):
    raise NotImplementedError


def retrieval_distribution(query_vectors, document_vectors):
    raise NotImplementedError

### 구현 단계 2 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §2.2 Eq. (2): RAG-Sequence
- **노트북에서 구현할 부분:** Task 2 marginalization
- **구현 이유:** 원문의 확률 분해나 tensor 변환을 작은 입력에서 직접
  관찰하고, 고수준 API 뒤에 핵심 알고리즘을 숨기지 않기 위해서다.
- **완료 증거:** finite NLL

아래 TODO 전에 입력 token/feature shape와 수식의 축을 먼저 확인한다.

In [ ]:
# TODO 2: retriever와 generator likelihood를 marginalize하는 RAGSequence를 구현하세요.
class RAGSequence:
    def __init__(self, documents, config):
        raise NotImplementedError

    def generator_likelihood(self, queries, document_ids):
        raise NotImplementedError

    def forward(self, retrieval_probability, queries):
        raise NotImplementedError

    def grounded_answer(self, document_index):
        """Document id가 포함된 grounded answer를 반환한다."""
        raise NotImplementedError

### 구현 단계 3 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** Fig. 1: retriever+generator
- **노트북에서 구현할 부분:** Task 3
- **구현 이유:** 원문의 확률 분해나 tensor 변환을 작은 입력에서 직접
  관찰하고, 고수준 API 뒤에 핵심 알고리즘을 숨기지 않기 위해서다.
- **완료 증거:** recall@1/cited answer

아래 TODO 전에 입력 token/feature shape와 수식의 축을 먼저 확인한다.

In [ ]:
# TODO 3: marginal NLL, recall@1, grounded answer 평가를 구현하세요.
def rag_sequence_loss(sequence_probability):
    raise NotImplementedError


def evaluate_retrieval(similarity, documents, queries):
    raise NotImplementedError


def build_grounded_answers(model, top_indices):
    raise NotImplementedError

## 실험 기록

1. 핵심 수식의 기호와 코드 변수 이름을 대응시키세요.
2. 어떤 assertion이 가장 중요한 구현 오류를 잡는지 설명하세요.
3. 이 미니 데이터로 검증할 수 없는 원 논문의 주장을 하나 적으세요.
4. seed를 유지하고 변수 하나만 바꾼 뒤 metric 전후를 기록하세요.